# GPT-2 et BERT préentraînés

Ces deux modèles viennent de Hugging Face. **On ne les entraîne pas.**
Leurs poids ont déjà été appris sur de très grands corpus. Ici, on les interroge
seulement pour voir deux usages du Transformer.

| | GPT-2 | BERT multilingue |
|---|---|---|
| Classe | décodeur seul | encodeur seul |
| Attention | causale : pas de regard vers le futur | bidirectionnelle |
| Tâche de cette démo | continuer un texte | retrouver un mot masqué |
| Modèle | `gpt2` | `bert-base-multilingual-cased` |

GPT-2 a surtout vu de l'anglais, d'où la phrase d'amorce en anglais du TP.
BERT `bert-base-multilingual-cased` couvre le français et distingue les majuscules.
        

In [1]:
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.training import count_parameters, get_device, set_seed

set_seed(42)
device = get_device()
print("Dossier du projet :", ROOT)
print("Périphérique :", device)

import os
os.environ.setdefault("HF_HOME", str(ROOT / "outputs" / "huggingface"))
print("Cache Hugging Face :", os.environ["HF_HOME"])
        

Dossier du projet : /Users/hmid/Downloads/Deep-Learning-Transformers
Périphérique : mps
Cache Hugging Face : /Users/hmid/Downloads/Deep-Learning-Transformers/outputs/huggingface


## 1. GPT-2, décodeur seul

Le texte de départ est exactement :

`Artificial intelligence can help people`

On demande 50 **nouveaux** tokens, avec un tirage aléatoire (`do_sample=True`),
une température de 0,8 et un `top_p` de 0,9. Le `top_p` garde le plus petit
ensemble de tokens dont la probabilité cumulée atteint au moins 90 %.

GPT-2 n'a pas de token de padding dédié. On réutilise le token de fin, comme
dans le notebook du cours. Avec une seule phrase, ce padding ne sert presque pas.
        

In [2]:
import json
from pathlib import Path

statut = {"gpt2": None, "bert": None}
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    nom = "gpt2"
    tokenizer_gpt = AutoTokenizer.from_pretrained(nom)
    modele_gpt = AutoModelForCausalLM.from_pretrained(nom)
    tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
    modele_gpt.config.pad_token_id = tokenizer_gpt.pad_token_id
    modele_gpt.eval()

    texte = "Artificial intelligence can help people"
    entrees = tokenizer_gpt(texte, return_tensors="pt")
    print("Dimensions de l'entrée :", tuple(entrees["input_ids"].shape))
    set_seed(42)
    with torch.no_grad():
        sortie = modele_gpt.generate(
            **entrees,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer_gpt.pad_token_id,
        )
    texte_genere = tokenizer_gpt.decode(sortie[0], skip_special_tokens=True)
    print("\nTexte généré :")
    print(texte_genere)
    statut["gpt2"] = {
        "ok": True,
        "forme_entree": list(entrees["input_ids"].shape),
        "texte": texte_genere,
    }
except Exception as erreur:
    statut["gpt2"] = {"ok": False, "erreur": f"{type(erreur).__name__}: {erreur}"}
    print("GPT-2 n'a pas pu être exécuté.")
    print(statut["gpt2"]["erreur"])
    print("Aucun texte n'est inventé à la place de cette génération.")
        

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

W1119 22:21:10.544000 48202 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


... (2 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Dimensions de l'entrée : (1, 6)



Texte généré :
Artificial intelligence can help people understand and understand a vast range of things, from our everyday lives to the everyday activities of others, and from our work. For this reason, we think it's important to understand what is and isn't possible.


### Pourquoi l'attention est causale

GPT-2 produit le token suivant à partir des tokens déjà écrits. Dans chaque bloc,
une position peut lire son passé et elle-même, pas l'avenir. C'est nécessaire :
au moment où l'on écrit un mot, la suite n'existe pas encore.

La génération est autorégressive. Le token choisi est ajouté à la séquence,
puis le modèle recalcule le suivant. Une température inférieure à 1 concentre
le tirage. Elle ne rend pas le texte vrai : elle le rend seulement moins dispersé.
        

## 2. BERT, encodeur seul

La phrase demandée est :

`Paris est la [MASK] de la France.`

Le token de masque n'est pas tapé au hasard : on utilise `tokenizer.mask_token`,
afin d'obtenir l'identifiant que le modèle a vraiment appris. Le tokenizer ajoute
aussi `[CLS]` et `[SEP]`.

On lit les logits à la position du masque, on applique un softmax sur tout le
vocabulaire, puis on affiche les cinq tokens les plus probables.
        

In [3]:
try:
    from transformers import AutoModelForMaskedLM, AutoTokenizer

    nom = "bert-base-multilingual-cased"
    tokenizer_bert = AutoTokenizer.from_pretrained(nom)
    modele_bert = AutoModelForMaskedLM.from_pretrained(nom)
    modele_bert.eval()

    phrase = f"Paris est la {tokenizer_bert.mask_token} de la France."
    entrees = tokenizer_bert(phrase, return_tensors="pt")
    print("Texte :", phrase)
    print("Token de masque :", tokenizer_bert.mask_token)
    print("Dimensions de l'entrée :", tuple(entrees["input_ids"].shape))
    position = (entrees["input_ids"][0] == tokenizer_bert.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad():
        logits = modele_bert(**entrees).logits[0, position, :]
    probabilites = torch.softmax(logits, dim=-1)
    top_probabilites, top_indices = torch.topk(probabilites, k=5)
    propositions = []
    print("\nPropositions de BERT :")
    for probabilite, indice in zip(top_probabilites, top_indices):
        token = tokenizer_bert.decode([indice.item()]).strip()
        phrase_complete = phrase.replace(tokenizer_bert.mask_token, token)
        print(f"{probabilite.item():.2%} | {phrase_complete}")
        propositions.append({"probabilite": float(probabilite.item()), "token": token, "phrase": phrase_complete})

    # La cellule de code du notebook d'origine utilise aussi cette variante.
    phrase_rabat = f"Rabat est la {tokenizer_bert.mask_token} Maroc."
    entrees_rabat = tokenizer_bert(phrase_rabat, return_tensors="pt")
    position_rabat = (entrees_rabat["input_ids"][0] == tokenizer_bert.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad():
        logits_rabat = modele_bert(**entrees_rabat).logits[0, position_rabat, :]
    probs_rabat, ids_rabat = torch.topk(torch.softmax(logits_rabat, dim=-1), k=5)
    print("\nVariante présente dans le code d'origine :")
    rabat = []
    for probabilite, indice in zip(probs_rabat, ids_rabat):
        token = tokenizer_bert.decode([indice.item()]).strip()
        complete = phrase_rabat.replace(tokenizer_bert.mask_token, token)
        print(f"{probabilite.item():.2%} | {complete}")
        rabat.append({"probabilite": float(probabilite.item()), "token": token, "phrase": complete})
    statut["bert"] = {"ok": True, "paris": propositions, "rabat": rabat}
except Exception as erreur:
    statut["bert"] = {"ok": False, "erreur": f"{type(erreur).__name__}: {erreur}"}
    print("BERT n'a pas pu être exécuté.")
    print(statut["bert"]["erreur"])
    print("Aucune proposition n'est inventée.")

destination = ROOT / "outputs" / "metrics" / "02_modeles_preentraines.json"
destination.write_text(json.dumps(statut, ensure_ascii=False, indent=2), encoding="utf-8")
print("Compte rendu :", destination)
        

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte : Paris est la [MASK] de la France.
Token de masque : [MASK]
Dimensions de l'entrée : (1, 10)



Propositions de BERT :
36.33% | Paris est la capitale de la France.
21.09% | Paris est la ville de la France.
3.16% | Paris est la Cité de la France.
1.67% | Paris est la préfecture de la France.
1.64% | Paris est la banlieue de la France.

Variante présente dans le code d'origine :
42.90% | Rabat est la capitale Maroc.
20.66% | Rabat est la du Maroc.
6.23% | Rabat est la de Maroc.
3.46% | Rabat est la ville Maroc.
1.67% | Rabat est la Rabat Maroc.
Compte rendu : /Users/hmid/Downloads/Deep-Learning-Transformers/outputs/metrics/02_modeles_preentraines.json


### Pourquoi l'attention est bidirectionnelle

Le masque est au milieu de la phrase. Pour proposer un mot, BERT peut s'appuyer
sur « Paris est la » **et** sur « de la France ». Un décodeur causal n'aurait
pas le droit de voir la fin. Cette liberté correspond à l'entraînement masqué :
le modèle apprend à remplir un trou, pas à continuer un texte.

Les probabilités affichées sont celles du softmax sur le vocabulaire entier.
Elles décrivent la préférence du modèle, pas une certitude factuelle.
Un token proposé peut aussi être un fragment de mot.
        

## Résultats mesurés de cette exécution

GPT-2 a été téléchargé et interrogé, sans entraînement. L'entrée a la forme `(1, 6)`. Le texte obtenu avec la graine 42 est :

> Artificial intelligence can help people understand and understand a vast range of things, from our everyday lives to the everyday activities of others, and from our work. For this reason, we think it's important to understand what is and isn't possible.

BERT, sur « Paris est la [MASK] de la France. », propose : capitale 36,33 %, ville 21,09 %, Cité 3,16 %, préfecture 1,67 %, banlieue 1,64 %. La phrase la plus probable est « Paris est la capitale de la France. ».

Sur la variante « Rabat est la [MASK] Maroc. », le premier token est aussi « capitale », à 42,90 %. Ces deux modèles restent des modèles préentraînés.